<a href="https://colab.research.google.com/github/Erjg1012/big-data-con-Spark-/blob/main/big_data_y_an%C3%A1lisis_de_sentimiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get -qq install openjdk-17-jdk-headless > /dev/null
!pip -q install pyspark

In [ ]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SentimentSparkML-Colab") \
    .master("local[*]") \
    .getOrCreate()

spark

In [ ]:
import os, urllib.request, zipfile

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00331/sentiment%20labelled%20sentences.zip"
zip_name = "sentiment_labelled_sentences.zip"
data_dir = "sls_data"

if not os.path.exists(zip_name):
    urllib.request.urlretrieve(url, zip_name)

if not os.path.exists(data_dir):
    os.makedirs(data_dir, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(data_dir)

!ls -R sls_data | head -n 50


sls_data:
__MACOSX
sentiment labelled sentences

sls_data/__MACOSX:
sentiment labelled sentences

sls_data/__MACOSX/sentiment labelled sentences:

sls_data/sentiment labelled sentences:
amazon_cells_labelled.txt
imdb_labelled.txt
readme.txt
yelp_labelled.txt


In [ ]:
with zipfile.ZipFile(zip_name, "r") as z:
    with z.open("sentiment labelled sentences/readme.txt") as f:
        print(f.read().decode("utf-8"))

This dataset was created for the Paper 'From Group to Individual Labels using Deep Features', Kotzias et. al,. KDD 2015
Please cite the paper if you want to use it :)

It contains sentences labelled with positive or negative sentiment, extracted from reviews of products, movies, and restaurants

Format:
sentence \t score \n


Details:
Score is either 1 (for positive) or 0 (for negative)	
The sentences come from three different websites/fields:

imdb.com
amazon.com
yelp.com

For each website, there exist 500 positive and 500 negative sentences. Those were selected randomly for larger datasets of reviews. 
We attempted to select sentences that have a clearly positive or negative connotaton, the goal was for no neutral sentences to be selected.



For the full datasets look:

imdb: Maas et. al., 2011 'Learning word vectors for sentiment analysis'
amazon: McAuley et. al., 2013 'Hidden factors and hidden topics: Understanding rating dimensions with review text'
yelp: Yelp dataset challenge 

In [ ]:
with zipfile.ZipFile(zip_name, "r") as z:
    with z.open("sentiment labelled sentences/amazon_cells_labelled.txt") as f:
        for i in range(1000):
            print(f.readline().decode("utf-8").strip())


So there is no way for me to plug it in here in the US unless I go by a converter.	0
Good case, Excellent value.	1
Great for the jawbone.	1
Tied to charger for conversations lasting more than 45 minutes.MAJOR PROBLEMS!!	0
The mic is great.	1
I have to jiggle the plug to get it to line up right to get decent volume.	0
If you have several dozen or several hundred contacts, then imagine the fun of sending each of them one by one.	0
If you are Razr owner...you must have this!	1
Needless to say, I wasted my money.	0
What a waste of money and time!.	0
And the sound quality is great.	1
He was very impressed when going from the original battery to the extended battery.	1
If the two were seperated by a mere 5+ ft I started to notice excessive static and garbled sound from the headset.	0
Very good quality though	1
The design is very odd, as the ear "clip" is not very comfortable at all.	0
Highly recommend for any one who has a blue tooth phone.	1
I advise EVERYONE DO NOT BE FOOLED!	0
So Far So G

In [ ]:
from pyspark.sql.functions import col, lit, split, trim

base = os.path.join(data_dir, "sentiment labelled sentences")
paths = [
    (os.path.join(base, "amazon_cells_labelled.txt"), "amazon"),
    (os.path.join(base, "imdb_labelled.txt"), "imdb"),
    (os.path.join(base, "yelp_labelled.txt"), "yelp"),
]

dfs = []
for p, src in paths:
    raw = spark.read.text(p)
    parts = split(col("value"), "\t")
    df = raw.select(
        trim(parts.getItem(0)).alias("text"),
        parts.getItem(1).cast("int").alias("label")
    ).withColumn("source", lit(src))
    dfs.append(df)

data = dfs[0].unionByName(dfs[1]).unionByName(dfs[2])

data.show(10000, truncate=80)
data.groupBy("label").count().show()


+--------------------------------------------------------------------------------+-----+------+
|                                                                            text|label|source|
+--------------------------------------------------------------------------------+-----+------+
|So there is no way for me to plug it in here in the US unless I go by a conve...|    0|amazon|
|                                                     Good case, Excellent value.|    1|amazon|
|                                                          Great for the jawbone.|    1|amazon|
| Tied to charger for conversations lasting more than 45 minutes.MAJOR PROBLEMS!!|    0|amazon|
|                                                               The mic is great.|    1|amazon|
|      I have to jiggle the plug to get it to line up right to get decent volume.|    0|amazon|
|If you have several dozen or several hundred contacts, then imagine the fun o...|    0|amazon|
|                                     If

In [ ]:
train, test = data.randomSplit([0.8, 0.2], seed=42)
print("train:", train.count(), "test:", test.count())


train: 2439 test: 561


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF

# Etapas
tokenizer = RegexTokenizer(
    inputCol="text", outputCol="tokens",
    pattern="\\W+", minTokenLength=2
)
stop = StopWordsRemover(inputCol="tokens", outputCol="filtered")
hashing = HashingTF(inputCol="filtered", outputCol="tf", numFeatures=2**18)
idf = IDF(inputCol="tf", outputCol="features")

# Pipeline completo
pipe = Pipeline(stages=[tokenizer, stop, hashing, idf])

# Ajustar IDF con TODO el dataset y transformar TODO el dataset
model = pipe.fit(data)
result = model.transform(data)

# "Tabla" con todas las salidas por etapa (texto -> tokens -> filtered -> tf -> features)
result.select(
    "source",   # si tienes la columna source
    "label",    # etiqueta 0/1
    "text",
    "tokens",
    "filtered",
    "tf",
    "features"
).show(100, truncate=False)   # cambia 20 por el número de filas que quieras ver

+------+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
from pyspark.ml.classification import NaiveBayes

nb = NaiveBayes(featuresCol="features", labelCol="label", predictionCol="prediction")
nb_pipe = Pipeline(stages=[tokenizer, stop, hashing, idf, nb])

nb_model = nb_pipe.fit(train)
nb_pred = nb_model.transform(test)

nb_pred.select("text","label","prediction","probability").show(50, truncate=80)


+--------------------------------------------------------------------------------+-----+----------+-------------------------------------------+
|                                                                            text|label|prediction|                                probability|
+--------------------------------------------------------------------------------+-----+----------+-------------------------------------------+
|                                                             $50 Down the drain.|    0|       0.0| [0.9999996695967924,3.3040320757711646E-7]|
|.... Item arrived quickly and works great with my Metro PCS Samsung SCH-r450 ...|    1|       1.0|[3.5185308169562686E-16,0.9999999999999996]|
|                                                      2 thumbs up to this seller|    1|       1.0|  [3.327349930870288E-10,0.999999999667265]|
|          A lot of websites have been rating this a very good phone and so do I.|    1|       0.0| [0.9999999528596377,4.71403623275713

In [ ]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label", predictionCol="prediction", maxIter=50, regParam=0.0)
lr_pipe = Pipeline(stages=[tokenizer, stop, hashing, idf, lr])

lr_model = lr_pipe.fit(train)
lr_pred = lr_model.transform(test)

lr_pred.select("text","label","prediction","probability").show(50, truncate=80)


+--------------------------------------------------------------------------------+-----+----------+-------------------------------------------+
|                                                                            text|label|prediction|                                probability|
+--------------------------------------------------------------------------------+-----+----------+-------------------------------------------+
|                                                             $50 Down the drain.|    0|       1.0|   [0.11418365126000682,0.8858163487399932]|
|.... Item arrived quickly and works great with my Metro PCS Samsung SCH-r450 ...|    1|       1.0|                [4.390352491915744E-57,1.0]|
|                                                      2 thumbs up to this seller|    1|       0.0|  [0.999979842179256,2.0157820744048927E-5]|
|          A lot of websites have been rating this a very good phone and so do I.|    1|       0.0|                                  [1.

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def eval_metrics(pred_df, name="model"):
    ev_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    ev_prec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
    ev_rec = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
    ev_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

    print(f"\n== {name} ==")
    print(f"Accuracy:  {ev_acc.evaluate(pred_df):.4f}")
    print(f"Precision: {ev_prec.evaluate(pred_df):.4f}")
    print(f"Recall:    {ev_rec.evaluate(pred_df):.4f}")
    print(f"F1:        {ev_f1.evaluate(pred_df):.4f}")

eval_metrics(nb_pred, "NaiveBayes")
eval_metrics(lr_pred, "LogisticRegression")



== NaiveBayes ==
Accuracy:  0.7683
Precision: 0.7683
Recall:    0.7683
F1:        0.7683

== LogisticRegression ==
Accuracy:  0.7433
Precision: 0.7436
Recall:    0.7433
F1:        0.7432
